[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/process_tomography/process_tomography.ipynb)

# Learn a noisy quantum gate

Send known spin states through a device and count its detector outcomes: quantum process tomography recovers the map that acts on every state. Four directions arranged as a tetrahedron suffice for a qubit. Here we simulate their probabilities, recover rotation, dephasing and relaxation as extensors, and use the learned maps to predict other inputs and repeated uses.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from collections.abc import Iterator

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.quantum.process_tomography.core import sphere
from examples.quantum.process_tomography import render

np.set_printoptions(precision=4, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# A state is a scalar plus a vector, unchanged by reversal. A channel maps this whole space.
State = ga.gatype.self_reverse()                                          # 1 x y z
Channel = ga.gatype((State, State))                                       # State <- State
one = mv.scalar([1.0])                                                    # [] Scalar

# Every use of these devices applies the same fixed turn after its noise channel.
labels = ("Rotation", "Rotation + dephasing", "Rotation + relaxation")
angles = np.array([0.075, 0.075, 0.075])                                   # radians per use
phase_flip = np.array([0.0, 0.06, 0.0])
loss = np.array([0.0, 0.0, 0.10])
rotation_plane = (mv.yz + mv.zx + mv.xy) / np.sqrt(3)                     # [] Bivector

# Four calibration directions; six independent validation directions and their Bloch lengths.
directions = mv.vector([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]) / np.sqrt(3)  # [preparations] Vector
validation_directions = mv.vector([[1, 0, 0], [0, 1, 0], [0, 0, 1],
                                  [-1, 0, 0], [0, -1, 0], [0, 0, -1]])   # [heldout] Vector
bloch_lengths = np.array([1.0, 0.7, 0.3, 1.0, 0.7, 0.0])

latitudes = 12
longitudes = 2 * latitudes
sequence_steps = 48
frame_duration = 80
design_labels = ("Four tetrahedral probes", "Two opposite probes")

## 1. Prepare, evolve, measure

A state is `0.5 * (one + bloch)`: the Bloch vector points along the spin, with length one for a pure state and less for a mixture. The scalar part holds its normalization. Four detector effects, aligned with the tetrahedron, add to one; twice their scalar product with the outgoing state gives the four outcome probabilities. We use calibrated preparations and effects with exact probabilities, isolating noise in the gate.

In density-matrix notation these read as $\rho=\tfrac12(I+\mathbf r\cdot\boldsymbol\sigma)$, $F_b=\tfrac14(I+\mathbf n_b\cdot\boldsymbol\sigma)$, and $p(b\mid a)=\operatorname{tr}[F_b\,\mathcal E(\rho_a)]$.

In [ ]:
prepared = (one + directions) / 2                                         # [preparations] State
# Each effect is half a pure state. All four sum to one, so their probabilities sum to one.
effects = prepared / 2                                                    # [outcomes] State

# Phase noise has two unobserved alternatives: do nothing, or turn halfway in the xy plane.
# Their amplitudes have square-root weights. Their sandwiches add, not their amplitudes.
phase_paths = stack((one * np.sqrt(1 - phase_flip), mv.xy * np.sqrt(phase_flip)), axis=-1)  # [channels, paths] Rotor
dephasing = (phase_paths >> State).sum(axis=-1)                            # [channels] State <- State

north = (one + mv.z) / 2                                                  # [] State
south = (one - mv.z) / 2                                                  # [] State
# One relaxation path preserves north and attenuates south. The other transfers south to north.
# The scalar part of State lets this move the centre of the Bloch ball while remaining linear.
loss_paths = stack((north + south * np.sqrt(1 - loss),
                    (mv.x * south) * np.sqrt(loss)), axis=-1)             # [channels, paths] Scalar + Vector + Bivector
damping = (loss_paths >> State).sum(axis=-1)                               # [channels] State <- State

rotations = (rotation_plane * (-angles / 2)).exp()                        # [channels] Rotor
# Calling a map on a map composes them; the rotor then turns every output of the noisy map.
device = rotations >> damping(dephasing)                                  # [channels] State <- State
outputs = device[:, None](prepared)                                       # [channels, preparations] State
measured = 2 * effects.scalar_product(outputs[..., None])                  # [channels, preparations, outcomes] Scalar

render.draw_experiment(prepared, measured, labels);

## 2. Recover the outputs, then leave the input open

The tetrahedron's directions overlap. `dual_frame` sums their dyads into the overlap map and solves it against the samples to obtain the dual weights; this map is invertible because the samples span `State`. The detector's dual recovers the outputs, and the preparations' dual extends them to a `State <- State` channel. One sum of output–dual dyads constructs the complete channel, ready to apply to any state or compose with another channel. The dual weights need not be physical states.

In density-matrix notation, [Chuang and Nielsen's Eq. (3.3)](https://arxiv.org/pdf/quant-ph/9610001#page=2) records the same channel action:
$$
\mathcal E(\rho_j)=\sum_k\lambda_{jk}\rho_k.
$$
Their subsequent steps express that channel as a process matrix $\chi$: $\beta\chi=\lambda$, then $\chi=\kappa\lambda$. In tensor-index notation, the generalized-inverse condition [Eq. (3.7)](https://arxiv.org/pdf/quant-ph/9610001#page=3) reads
$$
\beta^{mn}_{jk}
= \sum_{st,xy} \beta^{st}_{jk}\,\kappa^{xy}_{st}\,\beta^{mn}_{xy}.
$$
The process matrix and the Kraus operators extracted in Eqs. (3.9–3.10) are other representations of the same channel. The reconstruction here retains the channel directly; it does not perform those representation conversions.

In [ ]:
def dual_frame(states: State) -> State:
    """Weights that undo the overlap between a spanning set of scalar-product readouts."""
    # Leaving State open makes each scalar product a readout on an arbitrary input.
    # Multiplication by states returns it along that state; summing gives the overlap map.
    overlap = (states * states.scalar_product(State)).sum(axis=-1)         # [...] State <- State
    return overlap[..., None].solve(states)                              # [..., samples] State


# The detector reads 2 * effects against the state, including the Born-rule normalization.
measurement_dual = dual_frame(2 * effects)                                 # [outcomes] State
recovered = (measured * measurement_dual).sum(axis=-1)                     # [channels, preparations] State

# Each recovered output gets the input readout belonging to its preparation.
# The open State slot is the input of the learned channel; only the probabilities enter this fit.
preparation_dual = dual_frame(prepared)                                   # [preparations] State
learned = (recovered * preparation_dual.scalar_product(State)).sum(axis=-1)  # [channels] State <- State

# Pure states fill the surface of the Bloch ball. The inferred channel carries them to its image.
surface = sphere(latitudes, longitudes)                                   # [latitudes + 1, longitudes + 1] State
images = learned[:, None, None](surface)                                  # [channels, latitudes + 1, longitudes + 1] State
probe_images = learned[:, None](prepared)                                 # [channels, preparations] State
render.draw_channels(surface, images, prepared, probe_images, labels);

## 3. Predict inputs the reconstruction has not seen

The reconstructed map predicts detector outcomes for any input, not just the four calibration states. These tests use axial states with several Bloch-vector lengths, including a completely mixed state. Dephasing leaves the centre of the ball fixed; relaxation moves it towards a preferred state. The scalar-to-vector part of the learned channel carries that displacement.

In Bloch-vector notation the channel reads as the affine map $\mathbf r'=A\mathbf r+\mathbf t$; the same operation is linear on the scalar-plus-vector state.

In [ ]:
# Length one gives a pure state, length zero the completely mixed state at the centre.
heldout = (one + validation_directions * bloch_lengths) / 2                # [heldout] State
actual = device[:, None](heldout)                                         # [channels, heldout] State
predicted = learned[:, None](heldout)                                     # [channels, heldout] State

# Compare measurable predictions on each held-out state, using the same detector.
observed_probabilities = 2 * effects.scalar_product(actual[..., None])     # [channels, heldout, outcomes] Scalar
predicted_probabilities = 2 * effects.scalar_product(predicted[..., None]) # [channels, heldout, outcomes] Scalar
render.draw_predictions(observed_probabilities, predicted_probabilities, labels);

## 4. Why four probes?

Two opposite preparations only test the scalar part and one spin direction. A device could scramble the transverse directions and still give the same answers on those two inputs. Their overlap map therefore has two missing directions; the tetrahedron spans the whole state space. Its four nonzero singular values are what make the reconstruction possible.

In [ ]:
complete = (prepared * prepared.scalar_product(State)).sum(axis=-1)       # [] State <- State
opposite = stack((north, south))                                          # [preparations] State
incomplete = (opposite * opposite.scalar_product(State)).sum(axis=-1)     # [] (1 + z) <- State

# The opposite probes produce only scalar and z outputs. Embed those in the whole State
# space so the square map's spectrum includes the two directions they leave undetermined.
spectra = stack((complete.svdvals(), incomplete.cast(Channel).svdvals()))  # [designs, directions] Scalar
render.print_completeness(spectra, design_labels)

## 5. Predict a sequence without measuring it again

Composition turns the reconstructed one-step map into the map for a whole sequence. A coherent turn keeps pure states on the sphere; noise contracts it, and relaxation also displaces it. The hollow points mark the four preparations, while the filled points follow their predicted outputs through repeated uses of each device.

In superoperator notation the sequence reads as $\widehat{\mathcal E}^{\,n}$.

In [ ]:
def powers(channel: Channel, steps: int) -> Iterator[Channel]:
    """One application, then two, three, and so on, as composed maps."""
    combined = channel                                                   # [channels] State <- State
    for _ in range(steps):
        yield combined
        combined = channel(combined)                                     # [channels] State <- State


# The input geometry stays fixed. Each frame contains only its two changing images.
frames = ((combined[:, None, None](surface), combined[:, None](prepared))
          for combined in powers(learned, sequence_steps))
movie = save_animation(render.animate(surface, prepared, frames, labels), "process_tomography", frame_duration)
display(Image(filename=str(movie)))

The reconstruction follows [Chuang and Nielsen, *Prescription for experimental determination of the dynamics of a quantum black box*](https://arxiv.org/abs/quant-ph/9610001); the tetrahedral detector is a qubit example of the [symmetric informationally complete measurements of Renes et al.](https://arxiv.org/abs/quant-ph/0310075). Here the probabilities are exact. Finite measurement counts introduce statistical uncertainty, and unconstrained linear inversion can then produce a map outside the physical set of quantum channels.